# ARC NeuroGolf static ONNX solver 09- localish_recolor

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
required={'onnx':'onnx','onnxruntime':'onnxruntime','onnxsim':'onnxsim','torch':'torch','numpy':'numpy'}
missing=[pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])


import json, os, random, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
import onnxruntime as ort
from onnxsim import simplify

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 75.5 MB/s eta 0:00:00


In [4]:
TASK_ID='task118'

ROOT=Path.cwd()

TASK_PATH=Path('/kaggle/input/competitions/neurogolf-2026/task118.json')
if not TASK_PATH.exists():
    TASK_PATH=ROOT/'task118.json'
OUTDIR=ROOT/'task118_static_work'
VALDIR=ROOT/'task118_structural_validation'
OUTDIR.mkdir(exist_ok=True)
VALDIR.mkdir(exist_ok=True)

In [5]:
class Task118PlusCompletion(nn.Module):
    def __init__(self):
        super().__init__()
        z=torch.zeros(1,1,30,30)
        self.register_buffer('z', z)
        self.register_buffer('z11', torch.zeros(1,1,1,1))
        self.register_buffer('bigneg', torch.full((1,1,30,30), -1e6))
        for k in [2,3]:
            size=2*k+1
            plus=torch.zeros(1,1,size,size)
            plus[0,0,k,:]=1.0
            plus[0,0,:,k]=1.0
            self.register_buffer(f'plus{k}', plus)
            hor=torch.zeros(1,1,1,size); hor[0,0,0,:]=1.0
            ver=torch.zeros(1,1,size,1); ver[0,0,:,0]=1.0
            self.register_buffer(f'hor{k}', hor)
            self.register_buffer(f'ver{k}', ver)

    def shift(self,t,dr:int,dc:int):
        # output[r,c] = input[r-dr,c-dc], zero outside 30x30
        if dr>0:
            t=torch.cat([self.z[:,:,:dr,:], t[:,:,:30-dr,:]], dim=2)
        elif dr<0:
            d=-dr
            t=torch.cat([t[:,:,d:,:], self.z[:,:,:d,:]], dim=2)
        if dc>0:
            t=torch.cat([self.z[:,:,:,:dc], t[:,:,:,:30-dc]], dim=3)
        elif dc<0:
            d=-dc
            t=torch.cat([t[:,:,:,d:], self.z[:,:,:,:d]], dim=3)
        return t

    def sample(self,t,dr:int,dc:int):
        # map at center gets t at center+(dr,dc)
        return self.shift(t, -dr, -dc)

    def active_rect(self, nonzero):
        row_has=(nonzero.sum(dim=3, keepdim=True)>0).float() # 1,1,30,1
        col_has=(nonzero.sum(dim=2, keepdim=True)>0).float() # 1,1,1,30
        acc=self.z11
        rows=[]
        for r in range(29,-1,-1):
            acc=(acc + row_has[:,:,r:r+1,:]).clamp(0,1)
            rows.append(acc)
        active_row=torch.cat(list(reversed(rows)), dim=2)
        acc=self.z11
        cols=[]
        for c in range(29,-1,-1):
            acc=(acc + col_has[:,:,:,c:c+1]).clamp(0,1)
            cols.append(acc)
        active_col=torch.cat(list(reversed(cols)), dim=3)
        return active_row * active_col

    def offsets(self,k:int):
        offs=[]
        for d in range(-k,k+1):
            offs.append((0,d))
        for d in range(-k,k+1):
            if d!=0: offs.append((d,0))
        return offs

    def pair_strong(self, red, k:int):
        hs=[]; vs=[]
        for a in range(-k,k+1):
            for b in range(a+1,k+1):
                if b-a>=k:
                    hs.append(self.sample(red,0,a)*self.sample(red,0,b))
                    vs.append(self.sample(red,a,0)*self.sample(red,b,0))
        h=hs[0]
        for x in hs[1:]: h=h+x
        v=vs[0]
        for x in vs[1:]: v=v+x
        return (h>0).float(), (v>0).float()

    def candidate_features(self, red, active_zero, k:int):
        plus=getattr(self,f'plus{k}'); hor=getattr(self,f'hor{k}'); ver=getattr(self,f'ver{k}')
        red_count=F.conv2d(red, plus, padding=k)
        zero_count=F.conv2d(active_zero, plus, padding=k)
        hcnt=F.conv2d(red, hor, padding=(0,k))
        vcnt=F.conv2d(red, ver, padding=(k,0))
        axis=((hcnt>0).float() * (vcnt>0).float())
        hp,vp=self.pair_strong(red,k)
        one_axis=((hp+vp)>0).float()
        valid=(zero_count<0.5).float() * (red_count>1.5).float() * ((axis+one_axis)>0).float()
        # edge red at exactly radius k
        edge=self.sample(red,-k,0)+self.sample(red,k,0)+self.sample(red,0,-k)+self.sample(red,0,k)
        edge=(edge>0).float()
        return valid, red_count, axis, edge

    def broadcast_max_to_cells(self, score2, score3):
        vals=[]
        for k,score in [(2,score2),(3,score3)]:
            for dr,dc in self.offsets(k):
                vals.append(self.shift(score, dr, dc))
        m=vals[0]
        for v in vals[1:]:
            m=torch.maximum(m,v)
        return m

    def selected_from_score(self, score, valid, red, maxcell, k:int):
        bad=self.z[:,:,:,:] * 0.0
        for dr,dc in self.offsets(k):
            r_at=self.sample(red,dr,dc)
            m_at=self.sample(maxcell,dr,dc)
            bad=bad + ((m_at > (score + 0.1)).float() * r_at)
        selected=valid * (bad < 0.5).float()
        return selected

    def forward(self,x):
        # x: [1,10,30,30] one-hot float
        red=x[:,2:3]
        five=x[:,5:6]
        zero=x[:,0:1]
        nonzero=(1.0-zero).clamp(0,1)
        active=self.active_rect(nonzero)
        active_zero=zero*active
        v2,rc2,axis2,edge2=self.candidate_features(red,active_zero,2)
        v3,rc3,axis3,edge3=self.candidate_features(red,active_zero,3)
        score20=v2*(rc2*100.0 + edge2*10.0 + axis2*5.0 - 2.0) + (1-v2)*self.bigneg
        score30=v3*(rc3*100.0 + edge3*10.0 + axis3*5.0 - 3.0) + (1-v3)*self.bigneg
        maxcell0=self.broadcast_max_to_cells(score20,score30)
        sel20=self.selected_from_score(score20,v2,red,maxcell0,2)
        sel30=self.selected_from_score(score30,v3,red,maxcell0,3)
        global3=(sel30*edge3).amax(dim=(2,3), keepdim=True) # [1,1,1,1]
        pref2=1.0-global3; pref3=global3
        score2=v2*(rc2*100.0 + pref2*20.0 + edge2*10.0 + axis2*5.0 - 2.0) + (1-v2)*self.bigneg
        score3=v3*(rc3*100.0 + pref3*20.0 + edge3*10.0 + axis3*5.0 - 3.0) + (1-v3)*self.bigneg
        maxcell=self.broadcast_max_to_cells(score2,score3)
        sel2=self.selected_from_score(score2,v2,red,maxcell,2)
        sel3=self.selected_from_score(score3,v3,red,maxcell,3)
        fill=self.z[:,:,:,:] * 0.0
        for k,sel in [(2,sel2),(3,sel3)]:
            for dr,dc in self.offsets(k):
                fill=torch.maximum(fill, self.shift(sel,dr,dc))
        fill=(fill>0.5).float()*five
        outs=[]
        for ch in range(10):
            if ch==5:
                outs.append(x[:,5:6]*(1-fill))
            elif ch==8:
                outs.append(x[:,8:9]+fill)
            else:
                outs.append(x[:,ch:ch+1])
        return torch.cat(outs, dim=1)

def onehot(grid):
    a=np.array(grid,dtype=np.int64); h,w=a.shape
    x=np.zeros((1,10,30,30),dtype=np.float32)
    for c in range(10): x[0,c,:h,:w]=(a==c)
    # outside remains color 0 one-hot
    x[0,0,h:,:]=1
    x[0,0,:,w:]=1
    return x

def decode(y, shape):
    return y[0].argmax(0)[:shape[0],:shape[1]]

def validate_torch(model,task):
    model.eval(); rep={}
    with torch.no_grad():
        for split in ['train','test','arc-gen']:
            right=total=cell_right=cell_total=0; first=None
            for i,ex in enumerate(task.get(split,[])):
                inp=np.array(ex['input']); exp=np.array(ex['output'])
                x=torch.from_numpy(onehot(inp))
                y=model(x).numpy()
                pred=decode(y, exp.shape)
                ok=np.array_equal(pred,exp)
                right+=int(ok); total+=1
                cell_right+=int((pred==exp).sum()); cell_total+=exp.size
                if not ok and first is None:
                    diff=np.argwhere(pred!=exp)
                    first={'index':i,'n_diff':int(len(diff)),'first_diffs':[(int(r),int(c),int(pred[r,c]),int(exp[r,c])) for r,c in diff[:20]]}
            rep[split]={'right':right,'total':total,'cell_right':cell_right,'cell_total':cell_total,'first_wrong':first}
    return rep

def validate_ort(sess,task):
    rep={}
    for split in ['train','test','arc-gen']:
        right=total=cell_right=cell_total=0; first=None
        for i,ex in enumerate(task.get(split,[])):
            inp=np.array(ex['input']); exp=np.array(ex['output'])
            y=sess.run(None, {'input':onehot(inp)})[0]
            pred=decode(y, exp.shape)
            ok=np.array_equal(pred,exp)
            right+=int(ok); total+=1
            cell_right+=int((pred==exp).sum()); cell_total+=exp.size
            if not ok and first is None:
                diff=np.argwhere(pred!=exp)
                first={'index':i,'n_diff':int(len(diff)),'first_diffs':[(int(r),int(c),int(pred[r,c]),int(exp[r,c])) for r,c in diff[:20]]}
        rep[split]={'right':right,'total':total,'cell_right':cell_right,'cell_total':cell_total,'first_wrong':first}
    return rep

def structural_features(ex):
    inp=np.array(ex['input']); out=np.array(ex['output']); h,w=inp.shape
    red=list(map(tuple,np.argwhere(inp==2)))
    blue=list(map(tuple,np.argwhere((inp==5)&(out==8))))
    # infer completed pluses from output
    S=set(map(tuple,np.argwhere((out==2)|(out==8))))
    def mplus(r,c,k):
        return {(rr,c) for rr in range(max(0,r-k),min(h,r+k+1))}|{(r,cc) for cc in range(max(0,c-k),min(w,c+k+1))}
    cand=[]
    for r in range(h):
        for c in range(w):
            for k in [2,3]:
                pts=mplus(r,c,k)
                if pts<=S and any(inp[p]==2 for p in pts): cand.append((len(pts),k,r,c,pts))
    rem=set(S); chosen=[]
    for size,k,r,c,pts in sorted(cand, reverse=True):
        if pts<=rem:
            chosen.append((k,r,c,pts)); rem-=pts
    radii=sorted(set(k for k,r,c,pts in chosen))
    centers=[(r,c) for k,r,c,pts in chosen]
    n=len(chosen)
    red_counts=tuple(sorted(len([p for p in pts if inp[p]==2]) for k,r,c,pts in chosen))
    blue_counts=tuple(sorted(len([p for p in pts if out[p]==8]) for k,r,c,pts in chosen))
    pos_bucket=tuple(sorted((r//5,c//5) for r,c in centers))
    return {
        'grid_shape':f'{h}x{w}',
        'height':h,'width':w,
        'radius_set':str(tuple(radii)),
        'n_pluses':n,
        'red_count_signature':str(red_counts),
        'blue_count_signature':str(blue_counts),
        'center_bucket_signature':str(pos_bucket),
        'has_edge_plus':int(any(r-k<0 or c-k<0 or r+k>=h or c+k>=w for k,r,c,pts in chosen)),
        'total_red':len(red),
        'total_added':len(blue),
        'zero_density_bucket':round(float((inp==0).mean()),1),
    }

def structural_validation(sess, task):
    import pandas as pd
    rows=[]
    for split in ['train','test','arc-gen']:
        for i,ex in enumerate(task.get(split,[])):
            inp=np.array(ex['input']); exp=np.array(ex['output'])
            pred=decode(sess.run(None,{'input':onehot(inp)})[0], exp.shape)
            feat=structural_features(ex)
            feat.update({'split':split,'index':i,'exact':int(np.array_equal(pred,exp))})
            rows.append(feat)
    df=pd.DataFrame(rows)
    per_path=VALDIR/'per_example_structural_features_and_accuracy.csv'; df.to_csv(per_path,index=False)
    group_rows=[]; hold_rows=[]
    train_df=df[df.split=='train']; arc_df=df[df.split=='arc-gen']
    feature_cols=['grid_shape','height','width','radius_set','n_pluses','red_count_signature','blue_count_signature','center_bucket_signature','has_edge_plus','total_red','total_added','zero_density_bucket']
    for col in feature_cols:
        for val,g in df.groupby(col):
            group_rows.append({'feature':col,'value':val,'n':len(g),'exact':int(g.exact.sum()),'accuracy':float(g.exact.mean())})
        seen=set(train_df[col].astype(str))
        hold=arc_df[~arc_df[col].astype(str).isin(seen)]
        if len(hold):
            hold_rows.append({'feature':col,'train_unique':int(train_df[col].nunique()),'arc_unique':int(arc_df[col].nunique()),'heldout_unique':int(hold[col].nunique()),'heldout_total':int(len(hold)),'heldout_exact':int(hold.exact.sum()),'heldout_accuracy':float(hold.exact.mean())})
    pd.DataFrame(group_rows).to_csv(VALDIR/'structural_group_accuracy.csv',index=False)
    pd.DataFrame(hold_rows).to_csv(VALDIR/'structural_holdout_summary_direct_features.csv',index=False)
    return hold_rows

In [6]:
task=json.load(open(TASK_PATH))
model=Task118PlusCompletion().eval()
# A full PyTorch pass is slow because the static graph is intentionally unrolled.
# Full exact validation is performed with ONNX Runtime below.
onnx_path=ROOT/'task118_static_graph.onnx'
dummy=torch.from_numpy(onehot(np.zeros((1,1),dtype=np.int64)))
torch.onnx.export(model, dummy, onnx_path, input_names=['input'], output_names=['output'], opset_version=13, do_constant_folding=True, dynamic_axes=None, dynamo=False)
import onnx, onnxruntime as ort
m=onnx.load(str(onnx_path)); onnx.checker.check_model(m)
ops={}

/tmp/ipykernel_16/2582384364.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy, onnx_path, input_names=['input'], output_names=['output'], opset_version=13, do_constant_folding=True, dynamic_axes=None, dynamo=False)


In [7]:
for n in m.graph.node: ops[n.op_type]=ops.get(n.op_type,0)+1
forbidden=[op for op in ['Loop','Scan','NonZero','Unique','Script','Function'] if ops.get(op,0)]
risk=[op for op in ['Shape','Range','Expand','Gather','ScatterND','ConstantOfShape','Resize','NonMaxSuppression'] if ops.get(op,0)]
print('size',onnx_path.stat().st_size,'ops',ops,'forbidden',forbidden,'risk',risk)
sess=ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
rep=validate_ort(sess,task)
print('ort validation',json.dumps(rep,indent=2))
assert rep['train']['right']==rep['train']['total']
assert rep['test']['right']==rep['test']['total']
assert rep['arc-gen']['right']==rep['arc-gen']['total']
assert onnx_path.stat().st_size < 1_400_000
assert not forbidden
# allow Max/CumSum/Slice/Concat/etc; reject known dynamic/scatter risks
assert not risk

size 149735 ops {'Constant': 832, 'Slice': 166, 'Sub': 9, 'Clip': 61, 'ReduceSum': 2, 'Greater': 54, 'Cast': 59, 'Add': 151, 'Concat': 99, 'Mul': 89, 'Conv': 8, 'Less': 5, 'Max': 64, 'ReduceMax': 1} forbidden [] risk []
ort validation {
  "train": {
    "right": 4,
    "total": 4,
    "cell_right": 1314,
    "cell_total": 1314,
    "first_wrong": null
  },
  "test": {
    "right": 1,
    "total": 1,
    "cell_right": 418,
    "cell_total": 418,
    "first_wrong": null
  },
  "arc-gen": {
    "right": 262,
    "total": 262,
    "cell_right": 86915,
    "cell_total": 86915,
    "first_wrong": null
  }
}


In [8]:
# split
arc=task['arc-gen']; idx=list(range(len(arc))); random.Random(0).shuffle(idx); testset=set(idx[:max(1,int(round(0.3*len(idx))))])
fit_e=fit_t=test_e=test_t=0
for i,ex in enumerate(arc):
    inp=np.array(ex['input']); exp=np.array(ex['output'])
    pred=decode(sess.run(None,{'input':onehot(inp)})[0], exp.shape)
    ok=int(np.array_equal(pred,exp))
    if i in testset: test_e+=ok; test_t+=1
    else: fit_e+=ok; fit_t+=1
split={'seed':0,'fit_exact':fit_e,'fit_total':fit_t,'test_exact':test_e,'test_total':test_t}
hold_rows=structural_validation(sess,task)
summary={'task':'task118','validation':rep,'split_report':split,'onnx':{'onnx_size_bytes':onnx_path.stat().st_size,'under_1_4mb':onnx_path.stat().st_size<1_400_000,'forbidden':forbidden,'dynamic_scatter_risk_ops':risk,'op_counts':ops,'input_shape':[1,10,30,30],'output_shape':[1,10,30,30]},'structural_holdouts':hold_rows}
(VALDIR/'task118_structural_validation_report.md').write_text('# task118 structural validation report\n\n```json\n'+json.dumps(summary,indent=2)+'\n```\n')
with open(ROOT/'task118_static_graph_summary.json','w') as f: json.dump(summary,f,indent=2)


In [9]:
# write generic submission.zip
with zipfile.ZipFile(ROOT/'submission.zip','w',zipfile.ZIP_DEFLATED) as zf: zf.write(onnx_path,'task118.onnx')

repzip=ROOT/'task118_structural_validation_report.zip'
if repzip.exists(): repzip.unlink()
subprocess.check_call(['zip','-r',str(repzip),str(VALDIR)], stdout=subprocess.DEVNULL)
print(json.dumps(summary,indent=2))

{
  "task": "task118",
  "validation": {
    "train": {
      "right": 4,
      "total": 4,
      "cell_right": 1314,
      "cell_total": 1314,
      "first_wrong": null
    },
    "test": {
      "right": 1,
      "total": 1,
      "cell_right": 418,
      "cell_total": 418,
      "first_wrong": null
    },
    "arc-gen": {
      "right": 262,
      "total": 262,
      "cell_right": 86915,
      "cell_total": 86915,
      "first_wrong": null
    }
  },
  "split_report": {
    "seed": 0,
    "fit_exact": 183,
    "fit_total": 183,
    "test_exact": 79,
    "test_total": 79
  },
  "onnx": {
    "onnx_size_bytes": 149735,
    "under_1_4mb": true,
    "forbidden": [],
    "dynamic_scatter_risk_ops": [],
    "op_counts": {
      "Constant": 832,
      "Slice": 166,
      "Sub": 9,
      "Clip": 61,
      "ReduceSum": 2,
      "Greater": 54,
      "Cast": 59,
      "Add": 151,
      "Concat": 99,
      "Mul": 89,
      "Conv": 8,
      "Less": 5,
      "Max": 64,
      "ReduceMax": 1
    },